In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip -P /content/
!unzip /content/glove.6B.zip -d /content/glove.6B/


--2025-07-16 14:43:29--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-07-16 14:43:29--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-07-16 14:43:30--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘/content/glove.6B.z

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout, Flatten
from sklearn.metrics import classification_report, confusion_matrix
import re
import string
import os



num_words = 10000
maxlen = 256
embedding_dim = 100



zip_file_path = "/content/glove.6B.zip"
glove_unzip_dir = "/content/glove.6B"
os.makedirs(glove_unzip_dir, exist_ok=True)

if not os.path.exists(zip_file_path):
    print(f"Error: GloVe zip file not found at {zip_file_path}.")
    print("Please ensure 'glove.6B.zip' was uploaded to Colab session storage (Files icon -> Upload icon) BEFORE running this cell.")
    exit()

print(f"Unzipping {zip_file_path} to {glove_unzip_dir}...")
!unzip -q "$zip_file_path" -d "$glove_unzip_dir"
print("Unzipping complete.")

GLOVE_FILE = os.path.join(glove_unzip_dir, 'glove.6B.100d.txt')

if os.path.exists(GLOVE_FILE):
    print(f"Successfully located GloVe file: {GLOVE_FILE}")
else:
    print(f"Error: {GLOVE_FILE} not found after unzipping. This might indicate an issue with the zip file content or unzipping process.")
    exit()

print(f"Loading IMDb dataset with {num_words} words and maxlen {maxlen}...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=num_words)
print("IMDb dataset loaded successfully.")
print(f"Training samples: {len(x_train)}")
print(f"Test samples: {len(x_test)}")



print("Padding sequences...")
x_train = pad_sequences(x_train, maxlen=maxlen, padding='post', truncating='post')
x_test = pad_sequences(x_test, maxlen=maxlen, padding='post', truncating='post')
print("Sequences padded.")



print(f"Loading GloVe embeddings from {GLOVE_FILE}...")
embeddings_index = {}
with open(GLOVE_FILE, encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs
print(f"Found {len(embeddings_index)} word vectors in GloVe.")

embedding_matrix = np.zeros((num_words, embedding_dim))
word_index = imdb.get_word_index()
for word, i in word_index.items():
    if i < num_words:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector
print("Embedding matrix created for IMDb vocabulary.")



print("Building the Bidirectional LSTM model with GloVe and Dropout...")
model = Sequential()

model.add(Embedding(input_dim=num_words,
                    output_dim=embedding_dim,
                    weights=[embedding_matrix],
                    input_length=maxlen,
                    trainable=True))


model.add(Bidirectional(LSTM(units=128, dropout=0.3, recurrent_dropout=0.3)))


model.add(Dropout(0.4))


model.add(Flatten())

model.add(Dense(units=1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()
print("Model built successfully.")



print("Training the model...")

history = model.fit(x_train, y_train, epochs=10, batch_size=64, validation_split=0.2)
print("Model training complete.")



print("Evaluating the model on the test set...")
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

y_pred_proba = model.predict(x_test)
y_pred = (y_pred_proba > 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))



word_index = imdb.get_word_index()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokenized_text = []
    for word in text.split():
        tokenized_text.append(word_index.get(word, 2) + 3)
    padded_sequence = pad_sequences([tokenized_text], maxlen=maxlen, padding='post', truncating='post')
    return padded_sequence

def predict_sentiment(review_text):
    processed_input = preprocess_text(review_text)
    prediction_proba = model.predict(processed_input)[0][0]
    sentiment = "Positive" if prediction_proba >= 0.5 else "Negative"
    print(f"\nReview: '{review_text}'")
    print(f"Predicted Probability of Positive Sentiment: {prediction_proba:.4f}")
    print(f"Predicted Sentiment: {sentiment}")
    return sentiment, prediction_proba


print("\n--- Testing with new reviews ---")

positive_review = "This movie was absolutely fantastic! I loved every minute of it. The acting was superb and the story was captivating."
predict_sentiment(positive_review)

negative_review = "This film was a complete disaster. The plot was boring, the characters were annoying, and I fell asleep halfway through."
predict_sentiment(negative_review)

mixed_review = "The movie had some good moments, but it was also quite slow at times. Overall, it was just okay."
predict_sentiment(mixed_review)

another_positive = "Highly recommend this masterpiece! A true cinematic achievement that will stay with you."
predict_sentiment(another_positive)

another_negative = "Waste of time and money. Avoid at all costs. Nothing redeeming about this production."
predict_sentiment(another_negative)


Unzipping /content/glove.6B.zip to /content/glove.6B...
replace /content/glove.6B/glove.6B.50d.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/glove.6B/glove.6B.100d.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/glove.6B/glove.6B.200d.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/glove.6B/glove.6B.300d.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
Unzipping complete.
Successfully located GloVe file: /content/glove.6B/glove.6B.100d.txt
Loading IMDb dataset with 10000 words and maxlen 256...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
IMDb dataset loaded successfully.
Training samples: 25000
Test samples: 25000
Padding sequences...
Sequences padded.
Loading GloVe embeddings from /content/glove.6B/glove.6B.100d.txt...
Found 400000 word vectors in GloVe.
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
Embedding matrix created for IMDb vocabulary.
Building the Bidirectional LSTM model with GloVe and Dropout...


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,000,000 (3.81 MB)

 Trainable params: 1,000,000 (3.81 MB)

 Non-trainable params: 0 (0.00 B)

Model built successfully.
Training the model...
Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 515s 2s/step - accuracy: 0.5075 - loss: 0.7000 - val_accuracy: 0.5912 - val_loss: 0.6709
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 577s 2s/step - accuracy: 0.6228 - loss: 0.6419 - val_accuracy: 0.8188 - val_loss: 0.4261
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 561s 2s/step - accuracy: 0.8227 - loss: 0.4117 - val_accuracy: 0.8666 - val_loss: 0.3364
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 560s 2s/step - accuracy: 0.8828 - loss: 0.2925 - val_accuracy: 0.8746 - val_loss: 0.3133
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 562s 2s/step - accuracy: 0.9061 - loss: 0.2344 - val_accuracy: 0.8760 - val_loss: 0.3205
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 544s 2s/step - accuracy: 0.9256 - loss: 0.1988 - val_accuracy: 0.8784 - val_loss: 0.3340
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 500s 2s/step - accuracy: 0.9402 - loss: 0.1692 - val_accuracy: 0.8738 - val_loss: 0.3511
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 504s 2s/s

('Negative', np.float32(0.0023380017))